In [1]:
import utils 
import numpy as np
import pickle
import json
import cv2

CARPETA_CALIB = "../data/calib"
CARPETA_CAPTURAS = "../data/captures"

/Users/pilarotero/Desktop/3 año/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
with open("../data/stereo_calibration.pkl", "rb") as f:
    calib_pkl = pickle.load(f)

with open("../data/stereo_maps.pkl", "rb") as f:
    mapas_pkl = pickle.load(f)

ancho, alto = calib_pkl["image_size"]
baseline_mm = np.linalg.norm(calib_pkl["T"])

print(f"Resolución de cada cámara: {ancho} × {alto} píxeles")
print(f"Baseline: {baseline_mm:.4f} mm")

parametros = [
    ("Matriz intrínseca izquierda K", "left_K"),
    ("Distorsión izquierda [k1, k2, p1, p2, k3]", "left_dist"),
    ("Matriz intrínseca derecha K", "right_K"),
    ("Distorsión derecha [k1, k2, p1, p2, k3]", "right_dist"),
    ("Rotación relativa R", "R"),
    ("Traslación relativa T (mm)", "T"),
    ("Matriz esencial E", "E"),
    ("Matriz fundamental F", "F"),
]

with np.printoptions(precision=5, suppress=False):
    for descripcion, clave in parametros:
        print(f"\n{descripcion}:")
        print(calib_pkl[clave])

Resolución de cada cámara: 1920 × 1080 píxeles
Baseline: 59.8382 mm

Matriz intrínseca izquierda K:
[[592.85926   0.      958.13416]
 [  0.      590.30506 558.32698]
 [  0.        0.        1.     ]]

Distorsión izquierda [k1, k2, p1, p2, k3]:
[[ 4.97460e-03 -2.41411e-02  2.04411e-05 -5.93168e-04  3.84636e-03]]

Matriz intrínseca derecha K:
[[592.85164   0.      950.28608]
 [  0.      590.4396  543.67339]
 [  0.        0.        1.     ]]

Distorsión derecha [k1, k2, p1, p2, k3]:
[[ 0.00232 -0.02265 -0.00031 -0.00114  0.00351]]

Rotación relativa R:
[[ 9.99974e-01 -3.94819e-04  7.20356e-03]
 [ 3.81765e-04  9.99998e-01  1.81341e-03]
 [-7.20427e-03 -1.81061e-03  9.99972e-01]]

Traslación relativa T (mm):
[[-5.98369e+01]
 [-1.09342e-02]
 [ 3.96611e-01]]

Matriz esencial E:
[[-7.26392e-05 -3.96591e-01 -1.16531e-02]
 [-3.44799e-02 -1.08498e-01  5.98381e+01]
 [-1.19097e-02 -5.98368e+01 -1.08430e-01]]

Matriz fundamental F:
[[-3.04383e-10 -1.66904e-06  9.03211e-04]
 [-1.45073e-07 -4.58475e-07

### Estimación del tamaño del cuadrado
<p style="text-align: justify; text-justify: inter-word; font-size: 16px;"> 
OpenCV necesita el tamaño real del cuadrado del tablero para que la calibración quede en milímetros. Como no lo conocíamos, calibramos primero con un cuadrado de <strong>1</strong> (sin unidades): así el baseline queda expresado en "cuadrados". Comparándolo con el baseline de la calibración de <code>calib-stereo</code> (59.838 mm), despejamos el lado del cuadrado en mm.
</p>

In [3]:
# Calibramos con cuadrado = 1: el baseline queda medido en "cuadrados"
obj_puntos_1, pts_izq_1, pts_der_1, img_size = utils.detectar_esquinas(CARPETA_CALIB, (9, 6), 1.0)
calib_unitaria = utils.calibracion_estereo(obj_puntos_1, pts_izq_1, pts_der_1, img_size)

baseline_cuadrados = np.linalg.norm(calib_unitaria["T"])   
baseline_mm = np.linalg.norm(calib_pkl["T"])            

print("Baseline en mm:", baseline_mm.round(4))
print(f"Baseline en cuadrados: {baseline_cuadrados:.4f}")

Descartamos ../data/calib/left_3.jpg porque no se detecto el tablero en ambas imagenes
39 pares válidos
Baseline en mm: 59.8382
Baseline en cuadrados: 2.1358


In [4]:
# Si 2.137 cuadrados = 59.8382 mm, entonces 1 cuadrado = 59.8382 / 2.137 mm
size_cuadrado_mm = baseline_mm / baseline_cuadrados
print(f"Tamaño del cuadrado: {size_cuadrado_mm:.2f} mm") 

Tamaño del cuadrado: 28.02 mm


<p style="text-align: justify; text-justify: inter-word; font-size: 16px;">
Con el tamaño estimado, repetimos la calibración en milímetros para obtener el error de reproyección RMS de cada cámara y del sistema estéreo, y guardamos los resultados en <code>resultados/calibracion_propia.json</code> para usarlos en el notebook principal.
</p>

In [5]:
objetos_puntos, pts_izq, pts_der, img_size = utils.detectar_esquinas(CARPETA_CALIB, (9, 6), size_cuadrado_mm)

Descartamos ../data/calib/left_3.jpg porque no se detecto el tablero en ambas imagenes
39 pares válidos


In [6]:
# Calibracion estereo: stereoCalibrate estima intrinsecos, distorsion, R y T en un solo paso, sin valores iniciales (flags = 0)
error_estereo, K1_t, D1_t, K2_t, D2_t, R_t, T_t, E_t, F_t = cv2.stereoCalibrate(
    objetos_puntos, pts_izq, pts_der, None, None, None, None, img_size, flags = 0)

print(f"RMS estéreo: {error_estereo:.3f} px | fx: {K1_t[0, 0]:.1f} / {K2_t[0, 0]:.1f} px | "
      f"baseline: {np.linalg.norm(T_t):.2f} mm")

RMS estéreo: 0.799 px | fx: 592.7 / 593.4 px | baseline: 59.84 mm


In [8]:
resultados_calibracion = {
    "tamano_cuadrado_mm": float(size_cuadrado_mm),
    "rms_estereo_px": float(error_estereo),
    "baseline_propio_mm": float(np.linalg.norm(T_t)),
}
with open("../resultados/calibracion_propia.json", "w") as f:
    json.dump(resultados_calibracion, f, indent = 4)